In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

import pandas as pd
import numpy as np

from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge

from src.config import (
    SAMPLE_SUBMISSION_PATH,
    SUBMISSION_DIR,
    TARGET,
    ID_COL,
    RANDOM_STATE,
    N_SPLITS
)

print("Project root:", PROJECT_ROOT)

In [ ]:
def rmse(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return np.sqrt(mse)

In [ ]:
PREDICTION_ROOT = PROJECT_ROOT / "predictions"

oof_files = sorted(PREDICTION_ROOT.glob("*/*_oof.csv"))
test_files = sorted(PREDICTION_ROOT.glob("*/*_test.csv"))

print("OOF file count:", len(oof_files))
print("Test file count:", len(test_files))

for path in oof_files:
    print("OOF:", path.relative_to(PROJECT_ROOT))

for path in test_files:
    print("TEST:", path.relative_to(PROJECT_ROOT))

In [ ]:
oof_predictions = {}
test_predictions = {}

y_true = None
train_ids = None
test_ids = None

for oof_path in oof_files:
    group_name = oof_path.parent.name
    model_name = oof_path.name.replace("_oof.csv", "")
    full_model_name = f"{group_name}__{model_name}"

    df_oof = pd.read_csv(oof_path)

    if y_true is None:
        y_true = df_oof["y_true"].values
        train_ids = df_oof[ID_COL].values
    else:
        assert np.allclose(y_true, df_oof["y_true"].values), f"y_true mismatch: {full_model_name}"
        assert np.all(train_ids == df_oof[ID_COL].values), f"train id mismatch: {full_model_name}"

    oof_predictions[full_model_name] = df_oof["oof_pred"].values


for test_path in test_files:
    group_name = test_path.parent.name
    model_name = test_path.name.replace("_test.csv", "")
    full_model_name = f"{group_name}__{model_name}"

    df_test = pd.read_csv(test_path)

    if test_ids is None:
        test_ids = df_test[ID_COL].values
    else:
        assert np.all(test_ids == df_test[ID_COL].values), f"test id mismatch: {full_model_name}"

    test_predictions[full_model_name] = df_test["test_pred"].values


print("Loaded OOF models:", len(oof_predictions))
print("Loaded TEST models:", len(test_predictions))

assert set(oof_predictions.keys()) == set(test_predictions.keys()), "OOF ve TEST model listeleri aynı değil!"

In [ ]:
common_models = sorted(set(oof_predictions.keys()) & set(test_predictions.keys()))

print("Common model count:", len(common_models))
common_models

In [ ]:
model_scores = []

for model_name in common_models:
    pred = np.clip(oof_predictions[model_name], 0, 10)
    score = rmse(y_true, pred)

    model_scores.append({
        "model": model_name,
        "rmse": score
    })

model_scores_df = pd.DataFrame(model_scores)
model_scores_df = model_scores_df.sort_values("rmse").reset_index(drop=True)

model_scores_df

In [ ]:
best_single_model = model_scores_df.loc[0, "model"]
best_single_rmse = model_scores_df.loc[0, "rmse"]

best_single_test_pred = np.clip(test_predictions[best_single_model], 0, 10)

submission_best_single = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: best_single_test_pred
})

SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

best_single_path = SUBMISSION_DIR / f"submission_best_single_{best_single_model}.csv"

submission_best_single.to_csv(best_single_path, index=False)

print("Best single model:", best_single_model)
print("Best single RMSE:", best_single_rmse)
print("Saved:", best_single_path)

display(submission_best_single.head())

In [ ]:
simple_blend_results = []

max_n = min(12, len(common_models))

for n in range(2, max_n + 1):
    selected_models = model_scores_df.head(n)["model"].tolist()

    blend_oof = np.mean(
        [oof_predictions[m] for m in selected_models],
        axis=0
    )

    blend_oof = np.clip(blend_oof, 0, 10)
    score = rmse(y_true, blend_oof)

    simple_blend_results.append({
        "n_models": n,
        "models": selected_models,
        "rmse": score
    })

simple_blend_results_df = pd.DataFrame(simple_blend_results)
simple_blend_results_df = simple_blend_results_df.sort_values("rmse").reset_index(drop=True)

simple_blend_results_df

In [ ]:
best_simple_row = simple_blend_results_df.iloc[0]
best_simple_models = best_simple_row["models"]
best_simple_rmse = best_simple_row["rmse"]

best_simple_test_pred = np.mean(
    [test_predictions[m] for m in best_simple_models],
    axis=0
)

best_simple_test_pred = np.clip(best_simple_test_pred, 0, 10)

submission_simple_blend = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: best_simple_test_pred
})

simple_blend_path = SUBMISSION_DIR / "submission_best_simple_blend.csv"

submission_simple_blend.to_csv(simple_blend_path, index=False)

print("Best simple blend RMSE:", best_simple_rmse)
print("Best simple blend models:")
for m in best_simple_models:
    print("-", m)

print("Saved:", simple_blend_path)

display(submission_simple_blend.head())

In [ ]:
candidate_models = model_scores_df.head(6)["model"].tolist()

candidate_models

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)

weighted_results = []

n_trials = 5000

for i in range(n_trials):
    weights = rng.dirichlet(np.ones(len(candidate_models)))

    blend_oof = np.zeros(len(y_true))

    for w, model_name in zip(weights, candidate_models):
        blend_oof += w * oof_predictions[model_name]

    blend_oof = np.clip(blend_oof, 0, 10)
    score = rmse(y_true, blend_oof)

    result = {
        "rmse": score,
    }

    for model_name, weight in zip(candidate_models, weights):
        result[model_name] = weight

    weighted_results.append(result)

weighted_results_df = pd.DataFrame(weighted_results)
weighted_results_df = weighted_results_df.sort_values("rmse").reset_index(drop=True)

weighted_results_df.head(20)

In [ ]:
best_weighted_row = weighted_results_df.iloc[0]
best_weighted_rmse = best_weighted_row["rmse"]

best_weighted_test_pred = np.zeros(len(test_ids))

print("Best weighted RMSE:", best_weighted_rmse)
print("Weights:")

for model_name in candidate_models:
    weight = best_weighted_row[model_name]
    print(model_name, ":", weight)

    best_weighted_test_pred += weight * test_predictions[model_name]

best_weighted_test_pred = np.clip(best_weighted_test_pred, 0, 10)

submission_weighted_blend = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: best_weighted_test_pred
})

weighted_blend_path = SUBMISSION_DIR / "submission_best_weighted_blend.csv"

submission_weighted_blend.to_csv(weighted_blend_path, index=False)

print("Saved:", weighted_blend_path)

display(submission_weighted_blend.head())

In [ ]:
stack_models = model_scores_df.head(8)["model"].tolist()

stack_train = pd.DataFrame({
    model_name: oof_predictions[model_name]
    for model_name in stack_models
})

stack_test = pd.DataFrame({
    model_name: test_predictions[model_name]
    for model_name in stack_models
})

display(stack_train.head())
display(stack_test.head())

In [ ]:
cv = KFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

stack_oof = np.zeros(len(y_true))
stack_test_folds = np.zeros((len(test_ids), N_SPLITS))
stack_fold_scores = []

for fold, (train_idx, valid_idx) in enumerate(cv.split(stack_train, y_true), start=1):
    X_stack_train = stack_train.iloc[train_idx]
    X_stack_valid = stack_train.iloc[valid_idx]

    y_stack_train = y_true[train_idx]
    y_stack_valid = y_true[valid_idx]

    stack_model = Ridge(alpha=1.0)

    stack_model.fit(X_stack_train, y_stack_train)

    valid_pred = stack_model.predict(X_stack_valid)
    valid_pred = np.clip(valid_pred, 0, 10)

    stack_oof[valid_idx] = valid_pred

    fold_rmse = rmse(y_stack_valid, valid_pred)
    stack_fold_scores.append(fold_rmse)

    test_pred = stack_model.predict(stack_test)
    test_pred = np.clip(test_pred, 0, 10)

    stack_test_folds[:, fold - 1] = test_pred

    print(f"Fold {fold} RMSE:", fold_rmse)

stack_rmse_mean = np.mean(stack_fold_scores)
stack_rmse_std = np.std(stack_fold_scores)
stack_oof_rmse = rmse(y_true, stack_oof)

print("Stacking CV RMSE:", stack_rmse_mean, "±", stack_rmse_std)
print("Stacking OOF RMSE:", stack_oof_rmse)

In [ ]:
stack_test_pred = stack_test_folds.mean(axis=1)
stack_test_pred = np.clip(stack_test_pred, 0, 10)

submission_stack = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: stack_test_pred
})

stack_path = SUBMISSION_DIR / "submission_stacking_ridge.csv"

submission_stack.to_csv(stack_path, index=False)

print("Saved:", stack_path)

display(submission_stack.head())

In [ ]:
final_comparison = []

final_comparison.append({
    "method": f"best_single_{best_single_model}",
    "rmse": best_single_rmse,
    "file": str(best_single_path)
})

final_comparison.append({
    "method": "best_simple_blend",
    "rmse": best_simple_rmse,
    "file": str(simple_blend_path)
})

final_comparison.append({
    "method": "best_weighted_blend",
    "rmse": best_weighted_rmse,
    "file": str(weighted_blend_path)
})

final_comparison.append({
    "method": "stacking_ridge",
    "rmse": stack_oof_rmse,
    "file": str(stack_path)
})

final_comparison_df = pd.DataFrame(final_comparison)
final_comparison_df = final_comparison_df.sort_values("rmse").reset_index(drop=True)

final_comparison_df

In [ ]:
best_method = final_comparison_df.loc[0, "method"]

print("Best method:", best_method)

if best_method.startswith("best_single"):
    final_submission = submission_best_single.copy()

elif best_method == "best_simple_blend":
    final_submission = submission_simple_blend.copy()

elif best_method == "best_weighted_blend":
    final_submission = submission_weighted_blend.copy()

elif best_method == "stacking_ridge":
    final_submission = submission_stack.copy()

else:
    raise ValueError("Unknown best method")

final_best_path = SUBMISSION_DIR / "submission_ensemble_final_best.csv"

final_submission.to_csv(final_best_path, index=False)

print("Saved final best submission:", final_best_path)
display(final_submission.head())

In [ ]:
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

print("Final submission shape:", final_submission.shape)
print("Sample submission shape:", sample_submission.shape)

print("\nFinal columns:")
print(final_submission.columns.tolist())

print("\nSample columns:")
print(sample_submission.columns.tolist())

display(final_submission.head())
display(sample_submission.head())